In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/pytorch)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
from typing import Any

import torch as tr
import torch.optim as optim

In [ ]:
def _equal(a, b):
    """
    Compare two numbers or two tensors.
    """

    if (isinstance(a, tr.Tensor) == True) and (isinstance(b, tr.Tensor) == False):
        b = tr.tensor(b)
    elif (isinstance(a, tr.Tensor) == False) and (isinstance(b, tr.Tensor) == True):
        a = tr.tensor(a)

    if (isinstance(a, tr.Tensor) == True) and (isinstance(b, tr.Tensor) == True):
        result = tr.equal(a, b)

        if isinstance(result, bool):
            return result
        else:
            return tr.all(result)
    else:
        return a == b
    

def _almost(a, b, atol=0.01):
    """
    Compare two numbers or two tensors for approximate equality, with a tolerance of `atol`. 
    """

    if (isinstance(a, tr.Tensor) == True) and (isinstance(b, tr.Tensor) == False):
        b = tr.tensor(b)
    elif (isinstance(a, tr.Tensor) == False) and (isinstance(b, tr.Tensor) == True):
        a = tr.tensor(a)

    if (isinstance(a, tr.Tensor) == True) and (isinstance(b, tr.Tensor) == True):
        return tr.all(tr.abs(a - b) <= atol)
    else:
        return abs(a - b) <= atol


def check_eq(a: Any, b: Any, atol=None):
    if atol is None:
        return _equal(a, b)
    else:
        return _almost(a, b, atol=atol)


def assert_eq(a: Any, b: Any, atol=None):
    if atol is None:
        if _equal(a, b) == False:
            raise AssertionError(f"Assertion {a} == {b} failed")
    else:
        if _almost(a, b, atol=atol) == False:
            raise AssertionError(f"Assertion {a} ~ {b} failed (atol={atol})")


def assert_ne(a: Any, b: Any):
    if (a != b) == False:
        raise AssertionError(f"Assertion {a} != {b} failed")
    

def assert_lt(a: Any, b: Any):
    if (a < b) == False:
        raise AssertionError(f"Assertion {a} < {b} failed")

    
def assert_le(a: Any, b: Any):
    if (a <= b) == False:
        raise AssertionError(f"Assertion {a} <= {b} failed")


def assert_gt(a: Any, b: Any):
    if (a > b) == False:
        raise AssertionError(f"Assertion {a} > {b} failed")


def assert_ge(a: Any, b: Any):
    if (a >= b) == False:
        raise AssertionError(f"Assertion {a} >= {b} failed")
    

In [ ]:
#
# Create a PyTorch tensor from a number or a list of numbers.
#

def T(x):
    """
    Returns a PyTorch tensor created from `x`.
    """

    if isinstance(x, tr.Tensor):
        return x
   
    return tr.tensor(x, dtype=tr.float32)


def test_T():
    assert_eq(T(1/3), 0.33333, atol=0.01)
    assert_eq(T([1/1, 1/2, 1/3]), tr.tensor([1, 0.5, 0.33333]), atol=0.01)

In [ ]:
#
# Return a histogram of a vector if integers as a tuple of (unique_values, counts).
#

def count(vector):
    if isinstance(vector, tr.Tensor) == False:
        vector = tr.tensor(vector, dtype=tr.long)

    return vector.unique(sorted=True, return_counts=True)


def test_count():
    actual = count([1, 1, 2, 1, 2, 3, 1, 2, 3, 4, 1, 2, 3, 4, 5])
    expected = (tr.tensor([1, 2, 3, 4, 5]), tr.tensor([5, 4, 3, 2, 1]))

    assert_eq(actual[0], expected[0])
    assert_eq(actual[1], expected[1])

In [ ]:
def repeat(fn, times=10, reduction=tr.mean):
    return reduction(tr.tensor([fn() for _ in range(times)]))

In [ ]:
def test_model(model, x, y, epochs=500, lr=0.1):
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        optimizer.zero_grad()

        if hasattr(model, 'CUSTOM_GRAD'):
            loss = model(x, y)
        else:
            p = model(x)
            loss = model.loss(p, y)

        loss.backward()
        optimizer.step()

    with tr.no_grad():
        return (model.predict(x) == y).mean(dtype=tr.float32).item()


In [ ]:
def test_logical_fn(model, bool_fn, samples=100, epochs=500, lr=0.1):
    x = (tr.randn(samples, 2, dtype=tr.float32) > 0).float()
    y = bool_fn(x[:, 0], x[:, 1]).float().unsqueeze(1)
    return test_model(model, x, y, epochs, lr=lr)

In [ ]:
if __name__ == "__main__":
    test_T()
    test_count()
